# Session 10: Using Ragas to Evaluate a RAG Application built with LangChain and LangGraph

In the following notebook, we'll be looking at how [Ragas](https://github.com/explodinggradients/ragas) can be helpful in a number of ways when looking to evaluate your RAG applications!

While this example is rooted in LangChain/LangGraph - Ragas is framework agnostic (you don't even need to be using a framework!).

## 🤝 Breakout Room #1
  - Task 1: Installing Required Libraries
  - Task 2: Set Environment Variables
  - Task 3: Synthetic Dataset Generation for Evaluation using Ragas
  - Task 4: Construct our RAG application
  - Task 5: Evaluating our Application with Ragas
  - Task 6: Making Adjustments and Re-Evaluating
  - ***Activity #1: Implement a Different Reranking Strategy***


## Task 1: Installing Required Libraries

If you have not already done so, install the required libraries using the uv package manager:
``` bash

uv sync

```


## Task 2: Set Environment Variables:

We'll also need to provide our API keys.
> NOTE: In addition to OpenAI's models, this notebook will be using Cohere's Reranker - please be sure to [sign-up for an API key!](https://docs.cohere.com/reference/about)

You have two options for supplying your API keys in this session:
- Use environment variables (see Prerequisite #2 in the README.md)
- Provide them via a prompt when the notebook runs

The following code will load all of the environment variables in your `.env`. Then, it checks for the two API keys we need. If they are not there, it will prompt you to provide them.

First, OpenAI's for our LLM/embedding model combination!

Second, Cohere's for our reranking


In [1]:
import os
from getpass import getpass
from dotenv import load_dotenv

load_dotenv()

if not os.environ.get("OPENAI_API_KEY"):
    os.environ["OPENAI_API_KEY"] = getpass("Please enter your OpenAI API key!")

if not os.environ.get("COHERE_API_KEY"):
    os.environ["COHERE_API_KEY"] = getpass("Please enter your Cohere API key!")

## Task 3: Synthetic Dataset Generation for Evaluation using Ragas

We wil be using Ragas to build out a set of synthetic test questions, references, and reference contexts. This is useful because it will allow us to find out how our system is performing.

> NOTE: Ragas is best suited for finding *directional* changes in your LLM-based systems. The absolute scores aren't comparable in a vacuum.

### Data Preparation

We'll prepare our data using the Health & Wellness Guide - a comprehensive resource covering exercise, nutrition, sleep, and stress management.

Next, let's load our data into a familiar LangChain format using the `TextLoader`.

In [2]:
from langchain_community.document_loaders import TextLoader

loader = TextLoader("data/HealthWellnessGuide.txt")
docs = loader.load()

### Knowledge Graph Based Synthetic Generation

Ragas uses a knowledge graph based approach to create data. This is extremely useful as it allows us to create complex queries rather simply. The additional testset complexity allows us to evaluate larger problems more effectively, as systems tend to be very strong on simple evaluation tasks.

Let's start by defining our `generator_llm` (which will generate our questions, summaries, and more), and our `generator_embeddings` which will be useful in building our graph.

### Abstracted SDG

The above method is the full process - but we can shortcut that using the provided abstractions!

This will generate our knowledge graph under the hood, and will - from there - generate our personas and scenarios to construct our queries.



In [3]:
from ragas.llms import LangchainLLMWrapper
from ragas.embeddings import LangchainEmbeddingsWrapper
from langchain_openai import ChatOpenAI
from langchain_openai import OpenAIEmbeddings
generator_llm = LangchainLLMWrapper(ChatOpenAI(model="gpt-4.1"))
generator_embeddings = LangchainEmbeddingsWrapper(OpenAIEmbeddings())

In [4]:
from ragas.testset import TestsetGenerator

generator = TestsetGenerator(llm=generator_llm, embedding_model=generator_embeddings)
dataset = generator.generate_with_langchain_docs(docs, testset_size=10)

Applying HeadlinesExtractor:   0%|          | 0/1 [00:00<?, ?it/s]

Applying HeadlineSplitter:   0%|          | 0/1 [00:00<?, ?it/s]

Applying SummaryExtractor:   0%|          | 0/1 [00:00<?, ?it/s]

Applying CustomNodeFilter:   0%|          | 0/4 [00:00<?, ?it/s]

Applying [EmbeddingExtractor, ThemesExtractor, NERExtractor]:   0%|          | 0/9 [00:00<?, ?it/s]

Applying [CosineSimilarityBuilder, OverlapScoreBuilder]:   0%|          | 0/2 [00:00<?, ?it/s]

Generating personas:   0%|          | 0/1 [00:00<?, ?it/s]

Generating Scenarios:   0%|          | 0/1 [00:00<?, ?it/s]

Generating Samples:   0%|          | 0/9 [00:00<?, ?it/s]

In [5]:
dataset.to_pandas()

,user_input,reference_contexts,reference,synthesizer_name
0,Wut is the Cat-Cow Strech for lowr back pain?,[The Personal Wellness Guide A Comprehensive R...,The Cat-Cow Stretch is done by starting on han...,single_hop_specifc_query_synthesizer
1,Wut are Shoulder Shrugs and how doo they help ...,[The Personal Wellness Guide A Comprehensive R...,Shoulder Shrugs are an exercise where you rais...,single_hop_specifc_query_synthesizer
2,What are partial crunches and how can they hel...,[The Personal Wellness Guide A Comprehensive R...,Partial crunches are performed by lying on you...,single_hop_specifc_query_synthesizer
3,As a wellness enthusiast aiming to optimize my...,[PART 3: SLEEP AND RECOVERY Chapter 7: The Sci...,Non-REM sleep refers to the stages of sleep th...,single_hop_specifc_query_synthesizer
4,What role does REM sleep play in supporting me...,[PART 3: SLEEP AND RECOVERY Chapter 7: The Sci...,REM sleep is a stage of sleep during which the...,single_hop_specifc_query_synthesizer
5,i hear valerian root good for sleep but how i ...,[PART 3: SLEEP AND RECOVERY Chapter 7: The Sci...,herbal teas such as chamomile or valerian root...,single_hop_specifc_query_synthesizer
6,What are the recommended elements of an effect...,[PART 5: BUILDING HEALTHY HABITS Chapter 13: T...,"According to Chapter 15, an effective evening ...",single_hop_specifc_query_synthesizer
7,What topics are covered in PART 6 of the welln...,[PART 5: BUILDING HEALTHY HABITS Chapter 13: T...,"PART 6 covers common health concerns, includin...",single_hop_specifc_query_synthesizer
8,wut is PART 7 bout?,[PART 5: BUILDING HEALTHY HABITS Chapter 13: T...,PART 7: LIFESTYLE AND WELLNESS covers topics l...,single_hop_specifc_query_synthesizer


## Task 4: Construct our RAG application

Now we'll construct our LangChain RAG, which we will be evaluating using the above created test data!

### R - Retrieval

Let's start with building our retrieval pipeline, which will involve loading the same data we used to create our synthetic test set above.

> NOTE: We need to use the same data - as our test set is specifically designed for this data.

In [6]:
loader = TextLoader("data/HealthWellnessGuide.txt")
docs = loader.load()

Now that we have our data loaded, let's split it into chunks!

In [7]:
from langchain.text_splitter import RecursiveCharacterTextSplitter

text_splitter = RecursiveCharacterTextSplitter(chunk_size=50, chunk_overlap=0)
split_documents = text_splitter.split_documents(docs)
len(split_documents)

447

### ❓ Question #1:

What is the purpose of the `chunk_overlap` parameter in the `RecursiveCharacterTextSplitter`?

##### Answer:

The chunk_overlap parameter controls how many characters/tokens of content are repeated between consecutive chunks. Its main purpose is to preserve context at chunk boundaries so that sentences or ideas spanning two chunks don't get cut off and lost - when set to a positive value, the end of one chunk overlaps with the beginning of the next. When set to 0, there is no repeated content between chunks, meaning we don't preserve cross-boundary context.



Next up, we'll need to provide an embedding model that we can use to construct our vector store.

In [8]:
from langchain_openai import OpenAIEmbeddings

embeddings = OpenAIEmbeddings(model="text-embedding-3-small")

Now we can build our in memory QDrant vector store.

In [9]:
from langchain_qdrant import QdrantVectorStore
from qdrant_client import QdrantClient
from qdrant_client.http.models import Distance, VectorParams

client = QdrantClient(":memory:")

client.create_collection(
    collection_name="use_case_data",
    vectors_config=VectorParams(size=1536, distance=Distance.COSINE),
)

vector_store = QdrantVectorStore(
    client=client,
    collection_name="use_case_data",
    embedding=embeddings,
)

We can now add our documents to our vector store.

In [10]:
_ = vector_store.add_documents(documents=split_documents)

Let's define our retriever.

In [11]:
retriever = vector_store.as_retriever(search_kwargs={"k": 3})

Now we can produce a node for retrieval!

In [12]:
def retrieve(state):
  retrieved_docs = retriever.invoke(state["question"])
  return {"context" : retrieved_docs}

### A - Augmented

Let's create a simple RAG prompt!

In [13]:
from langchain.prompts import ChatPromptTemplate

RAG_PROMPT = """\
You are a helpful assistant who answers questions based on provided context. You must only use the provided context, and cannot use your own knowledge.

### Question
{question}

### Context
{context}
"""

rag_prompt = ChatPromptTemplate.from_template(RAG_PROMPT)

### G - Generation

We'll also need an LLM to generate responses - we'll use `gpt-4o-nano` to avoid using the same model as our judge model.

In [14]:
from langchain_openai import ChatOpenAI

llm = ChatOpenAI(model="gpt-4.1-nano")

Then we can create a `generate` node!

In [15]:
def generate(state):
  docs_content = "\n\n".join(doc.page_content for doc in state["context"])
  messages = rag_prompt.format_messages(question=state["question"], context=docs_content)
  response = llm.invoke(messages)
  return {"response" : response.content}

### Building RAG Graph with LangGraph

Let's create some state for our LangGraph RAG graph!

In [16]:
from langgraph.graph import START, StateGraph
from typing_extensions import List, TypedDict
from langchain_core.documents import Document

class State(TypedDict):
  question: str
  context: List[Document]
  response: str

Now we can build our simple graph!

> NOTE: We're using `add_sequence` since we will always move from retrieval to generation. This is essentially building a chain in LangGraph.

In [17]:
graph_builder = StateGraph(State).add_sequence([retrieve, generate])
graph_builder.add_edge(START, "retrieve")
graph = graph_builder.compile()

Let's do a test to make sure it's doing what we'd expect.

In [18]:
response = graph.invoke({"question" : "What exercises help with lower back pain?"})

In [19]:
response["response"]

'The provided context does not specify any particular exercises that help with lower back pain.'

## Task 5: Evaluating our Application with Ragas

Now we can finally do our evaluation!

We'll start by running the queries we generated usign SDG above through our application to get context and responses.

In [20]:
for test_row in dataset:
  response = graph.invoke({"question" : test_row.eval_sample.user_input})
  test_row.eval_sample.response = response["response"]
  test_row.eval_sample.retrieved_contexts = [context.page_content for context in response["context"]]

In [21]:
dataset.samples[0].eval_sample.response

'The Cat-Cow Stretch for low back pain involves starting on your hands and knees, then alternating between arching your back up (cat) and...'

Then we can convert that table into a `EvaluationDataset` which will make the process of evaluation smoother.

In [22]:
from ragas import EvaluationDataset

evaluation_dataset = EvaluationDataset.from_pandas(dataset.to_pandas())

We'll need to select a judge model - in this case we're using the same model that was used to generate our Synthetic Data.

In [23]:
from ragas import evaluate
from ragas.llms import LangchainLLMWrapper

evaluator_llm = LangchainLLMWrapper(ChatOpenAI(model="gpt-4.1-mini"))

Next up - we simply evaluate on our desired metrics!

In [24]:
from ragas.metrics import LLMContextRecall, Faithfulness, FactualCorrectness, ResponseRelevancy, ContextEntityRecall, NoiseSensitivity
from ragas import evaluate, RunConfig

custom_run_config = RunConfig(timeout=360)

baseline_result = evaluate(
    dataset=evaluation_dataset,
    metrics=[LLMContextRecall(), Faithfulness(), FactualCorrectness(), ResponseRelevancy(), ContextEntityRecall(), NoiseSensitivity()],
    llm=evaluator_llm,
    run_config=custom_run_config
)
baseline_result

Evaluating:   0%|          | 0/54 [00:00<?, ?it/s]

{'context_recall': 0.2500, 'faithfulness': 0.6237, 'factual_correctness': 0.5111, 'answer_relevancy': 0.6329, 'context_entity_recall': 0.2443, 'noise_sensitivity_relevant': 0.0000}

## Task 6: Making Adjustments and Re-Evaluating

Now that we've got our baseline - let's make a change and see how the model improves or doesn't improve!




We'll first set our retriever to return more documents, which will allow us to take advantage of the reranking.

In [25]:
text_splitter = RecursiveCharacterTextSplitter(chunk_size=500, chunk_overlap=30)
split_documents = text_splitter.split_documents(docs)
len(split_documents)

embeddings = OpenAIEmbeddings(model="text-embedding-3-small")

client = QdrantClient(":memory:")

client.create_collection(
    collection_name="use_case_data_new_chunks",
    vectors_config=VectorParams(size=1536, distance=Distance.COSINE),
)

vector_store = QdrantVectorStore(
    client=client,
    collection_name="use_case_data_new_chunks",
    embedding=embeddings,
)

_ = vector_store.add_documents(documents=split_documents)

adjusted_example_retriever = vector_store.as_retriever(search_kwargs={"k": 20})

Reranking, or contextual compression, is a technique that uses a reranker to compress the retrieved documents into a smaller set of documents.

This is essentially a slower, more accurate form of semantic similarity that we use on a smaller subset of our documents.

In [27]:
from langchain.retrievers.contextual_compression import ContextualCompressionRetriever
from langchain_cohere import CohereRerank

def retrieve_adjusted(state):
  compressor = CohereRerank(model="rerank-v3.5")
  compression_retriever = ContextualCompressionRetriever(
    base_compressor=compressor, base_retriever=adjusted_example_retriever, search_kwargs={"k": 5}
  )
  retrieved_docs = compression_retriever.invoke(state["question"])
  return {"context" : retrieved_docs}

We can simply rebuild our graph with the new retriever!

In [28]:
class AdjustedState(TypedDict):
  question: str
  context: List[Document]
  response: str

adjusted_graph_builder = StateGraph(AdjustedState).add_sequence([retrieve_adjusted, generate])
adjusted_graph_builder.add_edge(START, "retrieve_adjusted")
adjusted_graph = adjusted_graph_builder.compile()

In [29]:
response = adjusted_graph.invoke({"question" : "How can I improve my sleep quality?"})
response["response"]

'To improve your sleep quality, focus on establishing good sleep hygiene habits. Maintain a consistent sleep schedule, even on weekends. Create a relaxing bedtime routine, such as reading, gentle stretching, or taking a warm bath. Keep your bedroom cool, dark, and quiet by using blackout curtains or a sleep mask, and set the room temperature between 65-68°F. Limit screen exposure 1-2 hours before bed and avoid caffeine after 2 PM. Exercise regularly, but not too close to bedtime, and avoid alcohol and heavy meals before going to sleep. Additionally, ensure your mattress and pillows are comfortable. Incorporating relaxation techniques like progressive muscle relaxation, herbal teas such as chamomile or valerian root, meditation, and deep breathing exercises can also help promote better sleep.'

In [30]:
import time
import copy

rerank_dataset = copy.deepcopy(dataset)

for test_row in rerank_dataset:
  response = adjusted_graph.invoke({"question" : test_row.eval_sample.user_input})
  test_row.eval_sample.response = response["response"]
  test_row.eval_sample.retrieved_contexts = [context.page_content for context in response["context"]]
  time.sleep(2) # To try to avoid rate limiting.

In [31]:
rerank_dataset.samples[0].eval_sample.response

'The Cat-Cow Stretch is an exercise where you start on your hands and knees, then alternate between arching your back up (like a cat) and letting it sag down (like a cow). This movement helps to alleviate low back pain. You should perform 10-15 repetitions of this stretch.'

In [32]:
rerank_evaluation_dataset = EvaluationDataset.from_pandas(rerank_dataset.to_pandas())

In [33]:
rerank_result = evaluate(
    dataset=rerank_evaluation_dataset,
    metrics=[LLMContextRecall(), Faithfulness(), FactualCorrectness(), ResponseRelevancy(), ContextEntityRecall(), NoiseSensitivity()],
    llm=evaluator_llm,
    run_config=custom_run_config
)
rerank_result

Evaluating:   0%|          | 0/54 [00:00<?, ?it/s]

{'context_recall': 0.7778, 'faithfulness': 0.7947, 'factual_correctness': 0.6544, 'answer_relevancy': 0.8397, 'context_entity_recall': 0.1824, 'noise_sensitivity_relevant': 0.0302}

### ❓ Question #2:

Which system performed better, on what metrics, and why?

##### Answer:

The adjusted system with reranking performed better across nearly every metric. Context recall jumped from 0.25 to 0.78, faithfulness improved from 0.62 to 0.79, factual correctness went from 0.51 to 0.65, and answer relevancy rose from 0.63 to 0.84. Context entity recall also decreased slightly from 0.24 to 0.18, and noise sensitivity remained near zero in both systems. These improvements came from three key changes: increasing chunk size from 50 to 500 characters so each chunk carries meaningful paragraphs with slight overlap to preserve boundary context, widening retrieval from k=3 to k=20 to cast a broader net, and applying Cohere reranking to compress those 20 documents down to the most relevant ones. This two-stage approach of fast vector search for recall followed by slower reranking for precision proved significantly more effective than the naive baseline.

### ❓ Question #3:

What are the benefits and limitations of using synthetic data generation for RAG evaluation? Consider both the practical advantages and potential pitfalls.

##### Answer:

The benefits of synthetic data generation for RAG evaluation include automating the creation of question-answer pairs directly from your source documents, eliminating the need for expensive manual annotation. RAGAS uses a knowledge graph based algorithm that builds relationships between document concepts and generates diverse question types based on those relationships, including multi-hop queries that stress-test your system beyond simple lookups. This allows you to systematically evaluate how production-ready your RAG application is. However, there are important limitations. Synthetic questions may not accurately reflect how real users phrase queries or what they actually ask about. The quality of the generated test set depends entirely on the LLM producing it, so any biases or gaps in that model carry through to your evaluation. RAGAS scores are also directional rather than absolute, meaning they are useful for comparing two versions of the same system against the same test set but not for comparing across different test sets or benchmarking against external standards. For production systems, synthetic evaluation should be complemented with real user feedback and human-annotated test cases.

### ❓ Question #4:

If you were building a production wellness assistant, which Ragas metrics would be most important to optimize for and why? Consider the healthcare/wellness domain specifically.

##### Answer:

For a production wellness assistant, faithfulness and factual correctness would be the most critical metrics to optimize. In the healthcare and wellness domain, providing inaccurate information could lead users to follow harmful exercise routines, take incorrect supplement dosages, or ignore serious symptoms. Faithfulness ensures the system only makes claims grounded in the retrieved context rather than hallucinating medical advice, while factual correctness verifies those claims align with the actual source material. Context recall is also essential as a foundational metric because if the system fails to retrieve the right information in the first place, even a perfectly faithful model cannot produce correct answers. Answer relevancy matters as well since users seeking health guidance need direct, on-topic responses rather than tangential information. Noise sensitivity should be kept low to prevent irrelevant retrieved content from contaminating medical recommendations. In short, the stakes of misinformation in wellness make groundedness and accuracy the top priorities.

## Activity #1: Implement a Different Reranking Strategy

In this activity, you'll experiment with different reranking parameters or strategies to see how they affect the evaluation metrics.

**Requirements:**
1. Modify the `retrieve_adjusted` function to use different parameters (e.g., change `k` values, try different top_n for reranking)
2. Or implement a different retrieval enhancement strategy (e.g., hybrid search, query expansion)
3. Run the evaluation and compare results with the baseline and reranking results above
4. Document your findings in the markdown cell below

In [36]:
## Hybrid Approach ##
# Implement your custom retrieval strategy here
# Example: modify retrieve_adjusted with different parameters

text_splitter = RecursiveCharacterTextSplitter(chunk_size=500, chunk_overlap=30)
split_documents = text_splitter.split_documents(docs)

embeddings = OpenAIEmbeddings(model="text-embedding-3-small")

client = QdrantClient(":memory:")
client.create_collection(
    collection_name="hybrid_search_data",
    vectors_config=VectorParams(size=1536, distance=Distance.COSINE),
)

vector_store = QdrantVectorStore(
    client=client,
    collection_name="hybrid_search_data",
    embedding=embeddings,
)

_ = vector_store.add_documents(documents=split_documents)

vector_retriever = vector_store.as_retriever(search_kwargs={"k": 10})

from langchain_community.retrievers import BM25Retriever
bm25_retriever = BM25Retriever.from_documents(split_documents, k=10)

from langchain.retrievers import EnsembleRetriever

ensemble_retriever = EnsembleRetriever(
    retrievers=[vector_retriever, bm25_retriever],
    weights=[0.5, 0.5]
)


def retrieve_custom(state):
    retrieved_docs = ensemble_retriever.invoke(state["question"])
    return {"context": retrieved_docs}

class CustomState(TypedDict):
    question: str
    context: List[Document]
    response: str

custom_graph_builder = StateGraph(CustomState).add_sequence([retrieve_custom, generate])
custom_graph_builder.add_edge(START, "retrieve_custom")
custom_graph = custom_graph_builder.compile()

import copy
import time

hybrid_dataset = copy.deepcopy(dataset)

for test_row in hybrid_dataset:
    response = custom_graph.invoke({"question": test_row.eval_sample.user_input})
    test_row.eval_sample.response = response["response"]
    test_row.eval_sample.retrieved_contexts = [context.page_content for context in response["context"]]
    time.sleep(2)

hybrid_evaluation_dataset = EvaluationDataset.from_pandas(hybrid_dataset.to_pandas())

hybrid_result = evaluate(
    dataset=hybrid_evaluation_dataset,
    metrics=[LLMContextRecall(), Faithfulness(), FactualCorrectness(), ResponseRelevancy(), ContextEntityRecall(), NoiseSensitivity()],
    llm=evaluator_llm,
    run_config=custom_run_config
)
hybrid_result

Evaluating:   0%|          | 0/54 [00:00<?, ?it/s]

Exception raised in Job[23]: TimeoutError()
Exception raised in Job[35]: TimeoutError()


{'context_recall': 0.8889, 'faithfulness': 0.7664, 'factual_correctness': 0.6533, 'answer_relevancy': 0.9515, 'context_entity_recall': 0.2972, 'noise_sensitivity_relevant': 0.1667}

### Activity #1 Findings:

*Document your findings here: What strategy did you try? How did it compare to the baseline and reranking results?*

For Activity 1, I implemented a hybrid search strategy using LangChain's EnsembleRetriever, which combines vector search with BM25 keyword search using Reciprocal Rank Fusion (RRF) with equal weights of 0.5 each. Vector search captures semantic meaning but can miss exact terminology, while BM25 matches specific keywords precisely but has no understanding of meaning. Combining them addresses both weaknesses. Compared to the baseline, the hybrid approach improved significantly across nearly every metric. Context recall jumped from 0.25 to 0.89, answer relevancy rose from 0.63 to 0.95, and context entity recall improved from 0.24 to 0.30. Compared to the Task 6 reranking approach, hybrid search performed better on context recall (0.89 vs 0.78), answer relevancy (0.95 vs 0.84), and context entity recall (0.30 vs 0.18), suggesting the BM25 component helped retrieve more of the right source material and specific entities. However, faithfulness was slightly lower (0.77 vs 0.79) and noise sensitivity increased notably from 0.03 to 0.17, meaning the hybrid approach retrieved some irrelevant content that occasionally misled the generation model. This makes sense because combining two retrievers with equal weight means more total documents are considered, increasing the chance of pulling in noise alongside relevant content. A potential improvement would be to add Cohere reranking on top of the hybrid retrieval to get the best of all three strategies: broad recall from the ensemble, then precision filtering from the reranker.

